# Figure 7: I/L sibling spectral similarity across instruments

This notebook extends the predicted branch to **multiple instruments**. It builds reproducible I/L sibling pairs, predicts MS2 spectra from Koina under several instrument configurations, scores original-vs-switched similarity with every metric, and overlays the distributions per instrument.

Adding a new instrument is a one-line entry in `make_predictions/instruments.py::INSTRUMENT_PRESETS`. No bulk raw-data download is needed; the only network dependency is Koina.

See `CLAUDE.md` sections 7 and 8 for the instrument-coupling inventory and the extension guide.

In [ ]:
import os
import sys

# Make the repo root importable from the notebooks/ directory.
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from make_predictions.instruments import INSTRUMENT_PRESETS, get_config
from make_predictions.il_pipeline import (
    load_peptides,
    build_il_pairs,
    run_comparison,
    summarize,
)

In [ ]:
# Available instrument presets. Add new instruments in make_predictions/instruments.py.
for key, cfg in sorted(INSTRUMENT_PRESETS.items()):
    print(f"{key:20s} model={cfg.intensity_model:30s} instrument={cfg.instrument_type} frag={cfg.fragmentation_type} ce={cfg.collision_energy}")

In [ ]:
# Sample reproducible I/L-containing peptides from the human proteome and build sibling pairs.
FASTA = "../fasta/UP000005640_9606.fasta"
SAMPLE_SIZE = 2000
SEED = 42

peptides = load_peptides(FASTA, sample_size=SAMPLE_SIZE, seed=SEED)
originals, switched = build_il_pairs(peptides, seed=SEED)
print(f"{len(originals)} I/L sibling pairs")

In [ ]:
# Choose the instruments to compare. Default spans the three manuscript instrument
# families: Orbitrap (UniSpec on Lumos), Orbitrap Astral (Prosit HCD, NCE 30),
# and timsTOF (AlphaPeptDeep). See make_predictions/instruments.py for all presets.
INSTRUMENTS = ["unispec_lumos", "astral_dda", "alphapept_timstof"]
configs = [get_config(name) for name in INSTRUMENTS]

scores = run_comparison(originals, switched, configs)
scores.head()

In [ ]:
# Mean similarity per metric per instrument.
CORE_METRICS = ["spectral_angle", "mse", "weighted_dot_product"]
summarize(scores, CORE_METRICS).round(4)

In [ ]:
# Overlay the similarity distributions per instrument.
metrics = [m for m in CORE_METRICS if m in scores.columns]
fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
if len(metrics) == 1:
    axes = [axes]
for ax, metric in zip(axes, metrics):
    for instrument, sub in scores.groupby("instrument"):
        values = sub[metric].dropna()
        if len(values) > 1:
            sns.kdeplot(values, ax=ax, label=instrument, fill=True, alpha=0.25)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    ax.legend(fontsize=8)
fig.suptitle("I/L sibling spectral similarity by instrument (predicted spectra)")
fig.tight_layout()
fig.savefig("../temp_data/figure7_multi_instrument.png", dpi=150)
plt.show()

## Adding a new instrument

1. Add an `InstrumentConfig` entry to `INSTRUMENT_PRESETS` in `make_predictions/instruments.py` with the correct Koina `intensity_model`, `instrument_type`, `fragmentation_type` and `collision_energy`. Check the model card at https://koina.wilhelmlab.org for the accepted enum values.
2. Add its key to the `INSTRUMENTS` list above and re-run.

For an **observed** dataset from a new instrument, use `scripts/download_dataset.py <accession>` to fetch processed peak lists, then conform them to the schemas in `CLAUDE.md` section 6 and reuse the annotation/scoring path from `figure3`-`figure6`.